<a href="https://colab.research.google.com/github/rkahrya1311/amazon-ml-challenge-2026/blob/arun%2Ffeatures/arun_feature.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ============================================================
# AMAZON ML 2026
# ARUN — SIMILARITY FEATURES
# DIRECT COLAB UPLOAD VERSION
# ============================================================

# ------------------------------------------------------------
# 1. Install libraries
# ------------------------------------------------------------

!pip install -q rapidfuzz scikit-learn


# ------------------------------------------------------------
# 2. Imports
# ------------------------------------------------------------

import os
import re
import string
import warnings

import numpy as np
import pandas as pd

from rapidfuzz.fuzz import ratio
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")


# ------------------------------------------------------------
# 3. Check uploaded files
# ------------------------------------------------------------

file_names = [
    "train_source1.tsv",
    "train_source2.tsv",
    "train_source3.tsv",
    "train_ground_truth.tsv"
]

print("=" * 70)
print("CHECKING COLAB UPLOADED FILES")
print("=" * 70)

for file_name in file_names:

    path = os.path.join("/content", file_name)

    if os.path.exists(path):
        print("✓", file_name)
    else:
        print("✗ MISSING:", file_name)

        raise FileNotFoundError(
            f"\n{file_name} was not found in /content/.\n"
            "Upload all four TSV files to Colab first."
        )


# ------------------------------------------------------------
# 4. Load datasets
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("LOADING DATASETS")
print("=" * 70)

s1 = pd.read_csv(
    "/content/train_source1.tsv",
    sep="\t"
)

s2 = pd.read_csv(
    "/content/train_source2.tsv",
    sep="\t"
)

s3 = pd.read_csv(
    "/content/train_source3.tsv",
    sep="\t"
)

ground_truth = pd.read_csv(
    "/content/train_ground_truth.tsv",
    sep="\t"
)

print("S1 shape:", s1.shape)
print("S2 shape:", s2.shape)
print("S3 shape:", s3.shape)
print("Ground truth shape:", ground_truth.shape)


# ------------------------------------------------------------
# 5. Show columns
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SOURCE 1 COLUMNS")
print("=" * 70)
print(s1.columns.tolist())

print("\n" + "=" * 70)
print("SOURCE 2 COLUMNS")
print("=" * 70)
print(s2.columns.tolist())

print("\n" + "=" * 70)
print("SOURCE 3 COLUMNS")
print("=" * 70)
print(s3.columns.tolist())

print("\n" + "=" * 70)
print("GROUND TRUTH COLUMNS")
print("=" * 70)
print(ground_truth.columns.tolist())


# ------------------------------------------------------------
# 6. Automatically find important columns
# ------------------------------------------------------------

def find_column(df, possible_names):

    # Exact match
    for name in possible_names:
        if name in df.columns:
            return name

    # Case-insensitive match
    lower_map = {
        str(col).lower(): col
        for col in df.columns
    }

    for name in possible_names:
        if name.lower() in lower_map:
            return lower_map[name.lower()]

    # Partial match
    for col in df.columns:

        col_lower = str(col).lower()

        for name in possible_names:

            if name.lower() in col_lower:
                return col

    return None


ID_NAMES = [
    "id",
    "business_id",
    "entity_id",
    "source_id",
    "record_id",
    "businessid",
    "entityid"
]

NAME_NAMES = [
    "name",
    "business_name",
    "company_name",
    "businessname",
    "companyname"
]

ADDRESS_NAMES = [
    "address",
    "business_address",
    "company_address",
    "location",
    "street_address"
]

COUNTRY_NAMES = [
    "country",
    "country_name",
    "countrycode",
    "country_code"
]


def detect_columns(df, source_name):

    id_col = find_column(
        df,
        ID_NAMES
    )

    name_col = find_column(
        df,
        NAME_NAMES
    )

    address_col = find_column(
        df,
        ADDRESS_NAMES
    )

    country_col = find_column(
        df,
        COUNTRY_NAMES
    )

    print(f"\n{source_name}")
    print("ID      :", id_col)
    print("NAME    :", name_col)
    print("ADDRESS :", address_col)
    print("COUNTRY :", country_col)

    if None in [
        id_col,
        name_col,
        address_col,
        country_col
    ]:

        print("\nWARNING: Some columns could not be detected.")
        print("Available columns:")
        print(df.columns.tolist())

    return {
        "id": id_col,
        "name": name_col,
        "address": address_col,
        "country": country_col
    }


print("\n" + "=" * 70)
print("DETECTING SOURCE COLUMNS")
print("=" * 70)

s1_cols = detect_columns(
    s1,
    "SOURCE 1"
)

s2_cols = detect_columns(
    s2,
    "SOURCE 2"
)

s3_cols = detect_columns(
    s3,
    "SOURCE 3"
)


# ------------------------------------------------------------
# 7. Stop if required columns are missing
# ------------------------------------------------------------

for source_name, cols in [
    ("SOURCE 1", s1_cols),
    ("SOURCE 2", s2_cols)
]:

    missing = [
        key
        for key, value in cols.items()
        if value is None
    ]

    if missing:

        raise ValueError(
            f"\nCould not detect {missing} in {source_name}.\n"
            f"Please check the printed column names."
        )


# ------------------------------------------------------------
# 8. Normalization
# ------------------------------------------------------------

def normalize_text(value):

    if pd.isna(value):
        return ""

    value = str(value).lower()

    # Replace punctuation with spaces
    value = value.translate(
        str.maketrans(
            string.punctuation,
            " " * len(string.punctuation)
        )
    )

    # Normalize spaces
    value = re.sub(
        r"\s+",
        " ",
        value
    ).strip()

    return value


def normalize_country(value):

    return normalize_text(value)


# ------------------------------------------------------------
# 9. Similarity functions
# ------------------------------------------------------------

def name_jaccard(text1, text2):

    text1 = normalize_text(text1)
    text2 = normalize_text(text2)

    if not text1 or not text2:
        return 0.0

    tokens1 = set(text1.split())
    tokens2 = set(text2.split())

    union = tokens1 | tokens2

    if not union:
        return 0.0

    return len(tokens1 & tokens2) / len(union)


def name_levenshtein(text1, text2):

    text1 = normalize_text(text1)
    text2 = normalize_text(text2)

    if not text1 or not text2:
        return 0.0

    return ratio(
        text1,
        text2
    ) / 100.0


def name_tfidf_cosine(text1, text2):

    text1 = normalize_text(text1)
    text2 = normalize_text(text2)

    if not text1 or not text2:
        return 0.0

    try:

        vectorizer = TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2)
        )

        matrix = vectorizer.fit_transform(
            [text1, text2]
        )

        return float(
            cosine_similarity(
                matrix[0:1],
                matrix[1:2]
            )[0][0]
        )

    except Exception:

        return 0.0


def address_jaccard(text1, text2):

    return name_jaccard(
        text1,
        text2
    )


def address_levenshtein(text1, text2):

    return name_levenshtein(
        text1,
        text2
    )


def address_tfidf_cosine(text1, text2):

    return name_tfidf_cosine(
        text1,
        text2
    )


def country_match(country1, country2):

    country1 = normalize_country(country1)
    country2 = normalize_country(country2)

    if not country1 or not country2:
        return 0

    return int(
        country1 == country2
    )


# ------------------------------------------------------------
# 10. Combined feature function
# ------------------------------------------------------------

def calculate_similarity_features(
    name1,
    name2,
    address1,
    address2,
    country1,
    country2
):

    return {

        "name_jaccard":
            name_jaccard(
                name1,
                name2
            ),

        "name_levenshtein":
            name_levenshtein(
                name1,
                name2
            ),

        "name_tfidf_cosine":
            name_tfidf_cosine(
                name1,
                name2
            ),

        "address_jaccard":
            address_jaccard(
                address1,
                address2
            ),

        "address_levenshtein":
            address_levenshtein(
                address1,
                address2
            ),

        "address_tfidf_cosine":
            address_tfidf_cosine(
                address1,
                address2
            ),

        "country_match":
            country_match(
                country1,
                country2
            )
    }


# ------------------------------------------------------------
# 11. Test functions
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TESTING SIMILARITY FUNCTIONS")
print("=" * 70)

test_result = calculate_similarity_features(

    "Amazon India Private Limited",
    "Amazon India Pvt Limited",

    "Chennai Tamil Nadu India",
    "Chennai, Tamil Nadu, India",

    "India",
    "India"
)

for key, value in test_result.items():
    print(f"{key:30s}: {value:.4f}")


# ------------------------------------------------------------
# 12. Generate sample candidate pairs
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("GENERATING SAMPLE FEATURE TABLE")
print("=" * 70)

# Start with 100 x 100 = 10,000 pairs.
# This is ONLY for testing Arun's feature functions.

SAMPLE_SIZE = min(
    100,
    len(s1),
    len(s2)
)

sample_s1 = s1.head(
    SAMPLE_SIZE
)

sample_s2 = s2.head(
    SAMPLE_SIZE
)

print("S1 sample size:", len(sample_s1))
print("S2 sample size:", len(sample_s2))
print(
    "Total pairs:",
    len(sample_s1) * len(sample_s2)
)


# ------------------------------------------------------------
# 13. Calculate all features
# ------------------------------------------------------------

feature_rows = []

for _, row1 in sample_s1.iterrows():

    for _, row2 in sample_s2.iterrows():

        features = calculate_similarity_features(

            row1[s1_cols["name"]],
            row2[s2_cols["name"]],

            row1[s1_cols["address"]],
            row2[s2_cols["address"]],

            row1[s1_cols["country"]],
            row2[s2_cols["country"]]
        )

        feature_rows.append({

            "s1_id":
                row1[s1_cols["id"]],

            "s2_id":
                row2[s2_cols["id"]],

            **features
        })


features_df = pd.DataFrame(
    feature_rows
)


# ------------------------------------------------------------
# 14. Display feature table
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FEATURE TABLE")
print("=" * 70)

print(
    "Shape:",
    features_df.shape
)

display(
    features_df.head(10)
)


# ------------------------------------------------------------
# 15. Check missing values
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MISSING VALUE CHECK")
print("=" * 70)

display(
    features_df.isnull().sum()
)


# ------------------------------------------------------------
# 16. Check feature statistics
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FEATURE STATISTICS")
print("=" * 70)

feature_columns = [

    "name_jaccard",
    "name_levenshtein",
    "name_tfidf_cosine",

    "address_jaccard",
    "address_levenshtein",
    "address_tfidf_cosine",

    "country_match"
]

display(
    features_df[
        feature_columns
    ].describe()
)


# ------------------------------------------------------------
# 17. Inspect ground truth
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("GROUND TRUTH")
print("=" * 70)

print(
    "Ground truth columns:",
    ground_truth.columns.tolist()
)

display(
    ground_truth.head(10)
)


# ------------------------------------------------------------
# 18. Save output
# ------------------------------------------------------------

OUTPUT_DIR = "/content/arun_similarity"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

feature_output = os.path.join(
    OUTPUT_DIR,
    "sample_similarity_features.csv"
)

features_df.to_csv(
    feature_output,
    index=False
)

summary_output = os.path.join(
    OUTPUT_DIR,
    "similarity_feature_summary.csv"
)

features_df[
    feature_columns
].describe().T.to_csv(
    summary_output
)


# ------------------------------------------------------------
# 19. Final report
# ------------------------------------------------------------

print("\n")
print("=" * 70)
print("ARUN — SIMILARITY FEATURE TASK COMPLETED")
print("=" * 70)

print("""
✓ train_source1.tsv loaded
✓ train_source2.tsv loaded
✓ train_source3.tsv loaded
✓ train_ground_truth.tsv loaded

✓ Normalization completed

✓ name_jaccard()
✓ name_levenshtein()
✓ name_tfidf_cosine()

✓ address_jaccard()
✓ address_levenshtein()
✓ address_tfidf_cosine()

✓ country_match()

✓ Sample feature table generated
✓ Missing-value check completed
✓ Feature statistics generated

Output files:
""")

print(feature_output)
print(summary_output)

print("\nFeature table shape:", features_df.shape)

print("\n" + "=" * 70)
print("READY FOR KISHOR'S ML MODEL")
print("=" * 70)

CHECKING COLAB UPLOADED FILES
✓ train_source1.tsv
✓ train_source2.tsv
✓ train_source3.tsv
✓ train_ground_truth.tsv

LOADING DATASETS
S1 shape: (22060, 4)
S2 shape: (32494, 4)
S3 shape: (32941, 4)
Ground truth shape: (54759, 2)

SOURCE 1 COLUMNS
['entity_id', 'business_name', 'business_address', 'country']

SOURCE 2 COLUMNS
['entity_id', 'business_name', 'business_address', 'country']

SOURCE 3 COLUMNS
['entity_id', 'business_name', 'business_address', 'country']

GROUND TRUTH COLUMNS
['source1_entity_id', 'matched_entity_ids']

DETECTING SOURCE COLUMNS

SOURCE 1
ID      : entity_id
NAME    : business_name
ADDRESS : business_address
COUNTRY : country

SOURCE 2
ID      : entity_id
NAME    : business_name
ADDRESS : business_address
COUNTRY : country

SOURCE 3
ID      : entity_id
NAME    : business_name
ADDRESS : business_address
COUNTRY : country

TESTING SIMILARITY FUNCTIONS
name_jaccard                  : 0.6000
name_levenshtein              : 0.9231
name_tfidf_cosine             : 0.40

,s1_id,s2_id,name_jaccard,name_levenshtein,name_tfidf_cosine,address_jaccard,address_levenshtein,address_tfidf_cosine,country_match
0,S1-925783039,S2-166376419,0.0,0.080000,0.0,0.000000,0.400000,0.000000,0
1,S1-925783039,S2-764573417,0.0,0.318182,0.0,0.100000,0.474576,0.053551,1
2,S1-925783039,S2-639257739,0.0,0.093023,0.0,0.000000,0.246914,0.000000,0
3,S1-925783039,S2-163963287,0.0,0.137931,0.0,0.083333,0.281690,0.053551,1
4,S1-925783039,S2-49942811,0.0,0.255319,0.0,0.000000,0.307692,0.000000,1
5,S1-925783039,S2-138046867,0.0,0.484848,0.0,0.000000,0.356164,0.000000,1
6,S1-925783039,S2-584977605,0.0,0.224719,0.0,0.000000,0.315789,0.000000,0
7,S1-925783039,S2-277444929,0.0,0.311111,0.0,0.000000,0.285714,0.000000,0
8,S1-925783039,S2-721031885,0.0,0.212766,0.0,0.090909,0.382353,0.048185,1
9,S1-925783039,S2-508602797,0.0,0.241379,0.0,0.000000,0.314607,0.000000,0



MISSING VALUE CHECK


,0
s1_id,0
s2_id,0
name_jaccard,0
name_levenshtein,0
name_tfidf_cosine,0
address_jaccard,0
address_levenshtein,0
address_tfidf_cosine,0
country_match,0



FEATURE STATISTICS


,name_jaccard,name_levenshtein,name_tfidf_cosine,address_jaccard,address_levenshtein,address_tfidf_cosine,country_match
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,0.012944,0.307073,0.007978,0.011515,0.309884,0.006693,0.536000
std,0.049204,0.105884,0.032442,0.030737,0.084431,0.021030,0.498727
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.263158,0.000000,0.000000,0.269150,0.000000,0.000000
50%,0.000000,0.317460,0.000000,0.000000,0.313725,0.000000,1.000000
75%,0.000000,0.367347,0.000000,0.000000,0.357229,0.000000,1.000000
max,0.500000,0.835821,0.465292,0.428571,0.763636,0.479930,1.000000



GROUND TRUTH
Ground truth columns: ['source1_entity_id', 'matched_entity_ids']


,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."
5,S1-18727616,"S2-755677256,S3-187831601,S3-641489370,S3-4762..."
6,S1-318373630,"S2-660036492,S3-804600254"
7,S1-86989137,"S3-274817120,S3-312496301"
8,S1-29845983,"S2-648035184,S3-588502663"
9,S1-789009573,"S2-383871912,S3-74481402,S3-576451439"




ARUN — SIMILARITY FEATURE TASK COMPLETED

✓ train_source1.tsv loaded
✓ train_source2.tsv loaded
✓ train_source3.tsv loaded
✓ train_ground_truth.tsv loaded

✓ Normalization completed

✓ name_jaccard()
✓ name_levenshtein()
✓ name_tfidf_cosine()

✓ address_jaccard()
✓ address_levenshtein()
✓ address_tfidf_cosine()

✓ country_match()

✓ Sample feature table generated
✓ Missing-value check completed
✓ Feature statistics generated

Output files:

/content/arun_similarity/sample_similarity_features.csv
/content/arun_similarity/similarity_feature_summary.csv

Feature table shape: (10000, 9)

READY FOR KISHOR'S ML MODEL


In [4]:
# ============================================================
# ROBUST TSV LOADER
# ============================================================

import os
import pandas as pd

def load_tsv_robust(path):
    """
    Try several common encodings.
    """

    encodings = [
        "utf-8",
        "utf-8-sig",
        "cp1252",
        "latin1"
    ]

    last_error = None

    for encoding in encodings:

        try:

            df = pd.read_csv(
                path,
                sep="\t",
                encoding=encoding,
                low_memory=False
            )

            print(
                f"✓ Loaded {os.path.basename(path)} "
                f"using {encoding}"
            )

            return df

        except (UnicodeDecodeError, pd.errors.ParserError) as e:

            last_error = e

    raise RuntimeError(
        f"\nCould not read file: {path}\n"
        f"Last error: {last_error}"
    )


# ------------------------------------------------------------
# Load all four files
# ------------------------------------------------------------

print("=" * 70)
print("LOADING DATASETS")
print("=" * 70)

s1 = load_tsv_robust(
    "/content/train_source1.tsv"
)

s2 = load_tsv_robust(
    "/content/train_source2.tsv"
)

s3 = load_tsv_robust(
    "/content/train_source3.tsv"
)

ground_truth = load_tsv_robust(
    "/content/train_ground_truth.tsv"
)


# ------------------------------------------------------------
# Print shapes
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DATASET SHAPES")
print("=" * 70)

print("S1:", s1.shape)
print("S2:", s2.shape)
print("S3:", s3.shape)
print("Ground Truth:", ground_truth.shape)


# ------------------------------------------------------------
# Print columns
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("COLUMNS")
print("=" * 70)

print("\nS1:")
print(s1.columns.tolist())

print("\nS2:")
print(s2.columns.tolist())

print("\nS3:")
print(s3.columns.tolist())

print("\nGround Truth:")
print(ground_truth.columns.tolist())


# ------------------------------------------------------------
# Show sample rows
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SAMPLE ROWS")
print("=" * 70)

print("\nS1:")
display(s1.head(5))

print("\nS2:")
display(s2.head(5))

print("\nS3:")
display(s3.head(5))

print("\nGround Truth:")
display(ground_truth.head(5))

LOADING DATASETS
✓ Loaded train_source1.tsv using utf-8
✓ Loaded train_source2.tsv using utf-8
✓ Loaded train_source3.tsv using utf-8
✓ Loaded train_ground_truth.tsv using utf-8

DATASET SHAPES
S1: (110133, 4)
S2: (118845, 4)
S3: (142914, 4)
Ground Truth: (200210, 2)

COLUMNS

S1:
['entity_id', 'business_name', 'business_address', 'country']

S2:
['entity_id', 'business_name', 'business_address', 'country']

S3:
['entity_id', 'business_name', 'business_address', 'country']

Ground Truth:
['source1_entity_id', 'matched_entity_ids']

SAMPLE ROWS

S1:


,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India



S2:


,entity_id,business_name,business_address,country
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India
3,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US
4,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US



S3:


,entity_id,business_name,business_address,country
0,S3-202863386,wilfordhancock.com,"Mack Rd, Haltom City, Texas",US
1,S3-859268022,International South Consultants Private Ltd,NaN,India
2,S3-22467283,LLC Moncada Léarning Center,"5780 Fawn Ct, Fort Worth, Texas",US
3,S3-671162755,Moyna's Coffee,"1 Ivanhoe Ave, PO Box 6009, Cincinnati, Ohio",US
4,S3-960981775,Pvt. EFS Print Ventures Ltd.,"Door No 183, 41St Cross, 22Nd Main 9Th Block J...",India



Ground Truth:


,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."


In [5]:
# ============================================================
# AMAZON ML 2026
# ARUN — SIMILARITY FEATURES
# EXACT DATASET VERSION
# ============================================================

!pip install -q rapidfuzz scikit-learn


# ============================================================
# 1. IMPORTS
# ============================================================

import os
import re
import string
import warnings

import numpy as np
import pandas as pd

from rapidfuzz.fuzz import ratio
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")


# ============================================================
# 2. FILE PATHS
# ============================================================

S1_FILE = "/content/train_source1.tsv"
S2_FILE = "/content/train_source2.tsv"
S3_FILE = "/content/train_source3.tsv"
GT_FILE = "/content/train_ground_truth.tsv"


# ============================================================
# 3. LOAD DATASETS
# ============================================================

print("=" * 70)
print("LOADING DATASETS")
print("=" * 70)


def load_tsv(path):

    encodings = [
        "utf-8",
        "utf-8-sig",
        "cp1252",
        "latin1"
    ]

    for encoding in encodings:

        try:

            df = pd.read_csv(
                path,
                sep="\t",
                encoding=encoding,
                low_memory=False
            )

            print(
                f"✓ {os.path.basename(path)} "
                f"loaded using {encoding}"
            )

            return df

        except UnicodeDecodeError:
            continue

    raise ValueError(
        f"Could not decode {path}"
    )


s1 = load_tsv(S1_FILE)
s2 = load_tsv(S2_FILE)
s3 = load_tsv(S3_FILE)
ground_truth = load_tsv(GT_FILE)


# ============================================================
# 4. DATASET INFORMATION
# ============================================================

print("\n" + "=" * 70)
print("DATASET SHAPES")
print("=" * 70)

print("S1:", s1.shape)
print("S2:", s2.shape)
print("S3:", s3.shape)
print("Ground Truth:", ground_truth.shape)


print("\n" + "=" * 70)
print("COLUMNS")
print("=" * 70)

print("S1:", s1.columns.tolist())
print("S2:", s2.columns.tolist())
print("S3:", s3.columns.tolist())
print("Ground Truth:", ground_truth.columns.tolist())


# ============================================================
# 5. EXACT COLUMN NAMES
# ============================================================

ID_COL = "entity_id"
NAME_COL = "business_name"
ADDRESS_COL = "business_address"
COUNTRY_COL = "country"

GT_S1_COL = "source1_entity_id"
GT_MATCH_COL = "matched_entity_ids"


# ============================================================
# 6. NORMALIZATION
# ============================================================

def normalize_text(value):

    if pd.isna(value):
        return ""

    value = str(value).lower()

    # Replace punctuation with spaces
    value = value.translate(
        str.maketrans(
            string.punctuation,
            " " * len(string.punctuation)
        )
    )

    # Normalize whitespace
    value = re.sub(
        r"\s+",
        " ",
        value
    ).strip()

    return value


def normalize_country(value):

    return normalize_text(value)


# ============================================================
# 7. NAME JACCARD
# ============================================================

def name_jaccard(text1, text2):

    text1 = normalize_text(text1)
    text2 = normalize_text(text2)

    if not text1 or not text2:
        return 0.0

    tokens1 = set(text1.split())
    tokens2 = set(text2.split())

    union = tokens1 | tokens2

    if not union:
        return 0.0

    intersection = tokens1 & tokens2

    return len(intersection) / len(union)


# ============================================================
# 8. NAME LEVENSHTEIN
# ============================================================

def name_levenshtein(text1, text2):

    text1 = normalize_text(text1)
    text2 = normalize_text(text2)

    if not text1 or not text2:
        return 0.0

    return ratio(
        text1,
        text2
    ) / 100.0


# ============================================================
# 9. NAME TF-IDF COSINE
# ============================================================

def name_tfidf_cosine(text1, text2):

    text1 = normalize_text(text1)
    text2 = normalize_text(text2)

    if not text1 or not text2:
        return 0.0

    try:

        vectorizer = TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2)
        )

        matrix = vectorizer.fit_transform(
            [text1, text2]
        )

        score = cosine_similarity(
            matrix[0:1],
            matrix[1:2]
        )[0][0]

        return float(score)

    except Exception:

        return 0.0


# ============================================================
# 10. ADDRESS JACCARD
# ============================================================

def address_jaccard(text1, text2):

    return name_jaccard(
        text1,
        text2
    )


# ============================================================
# 11. ADDRESS LEVENSHTEIN
# ============================================================

def address_levenshtein(text1, text2):

    return name_levenshtein(
        text1,
        text2
    )


# ============================================================
# 12. ADDRESS TF-IDF COSINE
# ============================================================

def address_tfidf_cosine(text1, text2):

    return name_tfidf_cosine(
        text1,
        text2
    )


# ============================================================
# 13. COUNTRY MATCH
# ============================================================

def country_match(country1, country2):

    country1 = normalize_country(country1)
    country2 = normalize_country(country2)

    if not country1 or not country2:
        return 0

    return int(
        country1 == country2
    )


# ============================================================
# 14. ALL FEATURES TOGETHER
# ============================================================

def calculate_similarity_features(
    name1,
    name2,
    address1,
    address2,
    country1,
    country2
):

    return {

        "name_jaccard":
            name_jaccard(
                name1,
                name2
            ),

        "name_levenshtein":
            name_levenshtein(
                name1,
                name2
            ),

        "name_tfidf_cosine":
            name_tfidf_cosine(
                name1,
                name2
            ),

        "address_jaccard":
            address_jaccard(
                address1,
                address2
            ),

        "address_levenshtein":
            address_levenshtein(
                address1,
                address2
            ),

        "address_tfidf_cosine":
            address_tfidf_cosine(
                address1,
                address2
            ),

        "country_match":
            country_match(
                country1,
                country2
            )
    }


# ============================================================
# 15. TEST FUNCTIONS
# ============================================================

print("\n" + "=" * 70)
print("TESTING SIMILARITY FUNCTIONS")
print("=" * 70)

test_features = calculate_similarity_features(

    "Amazon India Private Limited",
    "Amazon India Pvt Limited",

    "Chennai Tamil Nadu India",
    "Chennai, Tamil Nadu, India",

    "India",
    "India"
)

for feature, value in test_features.items():

    print(
        f"{feature:30s}: {value:.4f}"
    )


# ============================================================
# 16. CREATE SMALL SAMPLE
# ============================================================

print("\n" + "=" * 70)
print("CREATING SAMPLE FEATURE DATA")
print("=" * 70)

# IMPORTANT:
# This is only for testing the similarity functions.
# Do NOT generate all 33K x 32K pairs.

SAMPLE_SIZE = 100

sample_s1 = s1.head(
    SAMPLE_SIZE
)

sample_s2 = s2.head(
    SAMPLE_SIZE
)

print(
    "S1 sample:",
    len(sample_s1)
)

print(
    "S2 sample:",
    len(sample_s2)
)

print(
    "Total pairs:",
    len(sample_s1) * len(sample_s2)
)


# ============================================================
# 17. CALCULATE FEATURES
# ============================================================

feature_rows = []

for _, row1 in sample_s1.iterrows():

    for _, row2 in sample_s2.iterrows():

        features = calculate_similarity_features(

            row1[NAME_COL],
            row2[NAME_COL],

            row1[ADDRESS_COL],
            row2[ADDRESS_COL],

            row1[COUNTRY_COL],
            row2[COUNTRY_COL]
        )

        feature_rows.append({

            "s1_entity_id":
                row1[ID_COL],

            "s2_entity_id":
                row2[ID_COL],

            **features
        })


features_df = pd.DataFrame(
    feature_rows
)


# ============================================================
# 18. DISPLAY FEATURE TABLE
# ============================================================

print("\n" + "=" * 70)
print("FEATURE TABLE")
print("=" * 70)

print(
    "Shape:",
    features_df.shape
)

display(
    features_df.head(10)
)


# ============================================================
# 19. CHECK MISSING VALUES
# ============================================================

print("\n" + "=" * 70)
print("MISSING VALUE CHECK")
print("=" * 70)

display(
    features_df.isnull().sum()
)


# ============================================================
# 20. FEATURE STATISTICS
# ============================================================

feature_columns = [

    "name_jaccard",
    "name_levenshtein",
    "name_tfidf_cosine",

    "address_jaccard",
    "address_levenshtein",
    "address_tfidf_cosine",

    "country_match"
]

print("\n" + "=" * 70)
print("FEATURE STATISTICS")
print("=" * 70)

display(
    features_df[
        feature_columns
    ].describe()
)


# ============================================================
# 21. PARSE GROUND TRUTH
# ============================================================

print("\n" + "=" * 70)
print("PARSING GROUND TRUTH")
print("=" * 70)


def parse_ground_truth(gt_df):

    pairs = []

    for _, row in gt_df.iterrows():

        s1_id = str(
            row[GT_S1_COL]
        ).strip()

        matched_ids = row[GT_MATCH_COL]

        if pd.isna(matched_ids):
            continue

        matched_ids = str(
            matched_ids
        ).strip()

        if not matched_ids:
            continue

        # Ground truth contains comma-separated S2/S3 IDs
        ids = [
            x.strip()
            for x in matched_ids.split(",")
            if x.strip()
        ]

        for matched_id in ids:

            pairs.append({

                "s1_entity_id":
                    s1_id,

                "matched_entity_id":
                    matched_id,

                "is_match":
                    1
            })

    return pd.DataFrame(
        pairs
    )


ground_truth_pairs = parse_ground_truth(
    ground_truth
)


print(
    "Ground-truth positive pairs:",
    len(ground_truth_pairs)
)

display(
    ground_truth_pairs.head(10)
)


# ============================================================
# 22. CHECK HOW MANY S2 MATCHES ARE IN SAMPLE
# ============================================================

sample_gt = features_df.merge(

    ground_truth_pairs,

    left_on=[
        "s1_entity_id",
        "s2_entity_id"
    ],

    right_on=[
        "s1_entity_id",
        "matched_entity_id"
    ],

    how="left"
)

sample_gt["is_match"] = (
    sample_gt["is_match"]
    .fillna(0)
    .astype(int)
)


print("\n" + "=" * 70)
print("SAMPLE GROUND-TRUTH DISTRIBUTION")
print("=" * 70)

print(
    sample_gt["is_match"]
    .value_counts()
)


# ============================================================
# 23. COMPARE FEATURES FOR MATCH / NON-MATCH
# ============================================================

print("\n" + "=" * 70)
print("MATCH VS NON-MATCH FEATURE COMPARISON")
print("=" * 70)

comparison = (
    sample_gt
    .groupby("is_match")[feature_columns]
    .mean()
)

display(
    comparison
)


# ============================================================
# 24. SAVE OUTPUT
# ============================================================

OUTPUT_DIR = "/content/arun_similarity"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


feature_file = os.path.join(
    OUTPUT_DIR,
    "sample_similarity_features.csv"
)

features_df.to_csv(
    feature_file,
    index=False
)


gt_file = os.path.join(
    OUTPUT_DIR,
    "ground_truth_pairs.csv"
)

ground_truth_pairs.to_csv(
    gt_file,
    index=False
)


comparison_file = os.path.join(
    OUTPUT_DIR,
    "feature_comparison.csv"
)

comparison.to_csv(
    comparison_file
)


# ============================================================
# 25. FINAL REPORT
# ============================================================

print("\n")
print("=" * 70)
print("ARUN — SIMILARITY TASK COMPLETED")
print("=" * 70)

print("""
DATASETS
✓ S1 loaded
✓ S2 loaded
✓ S3 loaded
✓ Ground truth loaded

SIMILARITY FEATURES
✓ name_jaccard
✓ name_levenshtein
✓ name_tfidf_cosine
✓ address_jaccard
✓ address_levenshtein
✓ address_tfidf_cosine
✓ country_match

VALIDATION
✓ Missing-value check
✓ Ground-truth parsing
✓ Match/non-match comparison

OUTPUT
""")

print(feature_file)
print(gt_file)
print(comparison_file)

print("\nFeature table shape:", features_df.shape)

print(
    "Ground-truth positive pairs:",
    len(ground_truth_pairs)
)

print("\n" + "=" * 70)
print("READY FOR ML INTEGRATION")
print("=" * 70)

LOADING DATASETS
✓ train_source1.tsv loaded using utf-8
✓ train_source2.tsv loaded using utf-8
✓ train_source3.tsv loaded using utf-8
✓ train_ground_truth.tsv loaded using utf-8

DATASET SHAPES
S1: (121150, 4)
S2: (129547, 4)
S3: (153910, 4)
Ground Truth: (218461, 2)

COLUMNS
S1: ['entity_id', 'business_name', 'business_address', 'country']
S2: ['entity_id', 'business_name', 'business_address', 'country']
S3: ['entity_id', 'business_name', 'business_address', 'country']
Ground Truth: ['source1_entity_id', 'matched_entity_ids']

TESTING SIMILARITY FUNCTIONS
name_jaccard                  : 0.6000
name_levenshtein              : 0.9231
name_tfidf_cosine             : 0.4030
address_jaccard               : 1.0000
address_levenshtein           : 1.0000
address_tfidf_cosine          : 1.0000
country_match                 : 1.0000

CREATING SAMPLE FEATURE DATA
S1 sample: 100
S2 sample: 100
Total pairs: 10000

FEATURE TABLE
Shape: (10000, 9)


,s1_entity_id,s2_entity_id,name_jaccard,name_levenshtein,name_tfidf_cosine,address_jaccard,address_levenshtein,address_tfidf_cosine,country_match
0,S1-925783039,S2-166376419,0.0,0.080000,0.0,0.000000,0.400000,0.000000,0
1,S1-925783039,S2-764573417,0.0,0.318182,0.0,0.100000,0.474576,0.053551,1
2,S1-925783039,S2-639257739,0.0,0.093023,0.0,0.000000,0.246914,0.000000,0
3,S1-925783039,S2-163963287,0.0,0.137931,0.0,0.083333,0.281690,0.053551,1
4,S1-925783039,S2-49942811,0.0,0.255319,0.0,0.000000,0.307692,0.000000,1
5,S1-925783039,S2-138046867,0.0,0.484848,0.0,0.000000,0.356164,0.000000,1
6,S1-925783039,S2-584977605,0.0,0.224719,0.0,0.000000,0.315789,0.000000,0
7,S1-925783039,S2-277444929,0.0,0.311111,0.0,0.000000,0.285714,0.000000,0
8,S1-925783039,S2-721031885,0.0,0.212766,0.0,0.090909,0.382353,0.048185,1
9,S1-925783039,S2-508602797,0.0,0.241379,0.0,0.000000,0.314607,0.000000,0



MISSING VALUE CHECK


,0
s1_entity_id,0
s2_entity_id,0
name_jaccard,0
name_levenshtein,0
name_tfidf_cosine,0
address_jaccard,0
address_levenshtein,0
address_tfidf_cosine,0
country_match,0



FEATURE STATISTICS


...


PARSING GROUND TRUTH
Ground-truth positive pairs: 756838


,s1_entity_id,matched_entity_id,is_match
0,S1-965667,S2-681193310,1
1,S1-965667,S2-743505751,1
2,S1-965667,S3-775321672,1
3,S1-965667,S3-11291185,1
4,S1-965667,S3-860443364,1
5,S1-55344266,S2-249013014,1
6,S1-55344266,S2-197070651,1
7,S1-55344266,S3-478195123,1
8,S1-55344266,S3-384364074,1
9,S1-343815751,S2-790675320,1



SAMPLE GROUND-TRUTH DISTRIBUTION
is_match
0    10000
Name: count, dtype: int64

MATCH VS NON-MATCH FEATURE COMPARISON


,name_jaccard,name_levenshtein,name_tfidf_cosine,address_jaccard,address_levenshtein,address_tfidf_cosine,country_match
is_match,,,,,,,
0,0.012944,0.307073,0.007978,0.011515,0.309884,0.006693,0.536




ARUN — SIMILARITY TASK COMPLETED

DATASETS
✓ S1 loaded
✓ S2 loaded
✓ S3 loaded
✓ Ground truth loaded

SIMILARITY FEATURES
✓ name_jaccard
✓ name_levenshtein
✓ name_tfidf_cosine
✓ address_jaccard
✓ address_levenshtein
✓ address_tfidf_cosine
✓ country_match

VALIDATION
✓ Missing-value check
✓ Ground-truth parsing
✓ Match/non-match comparison

OUTPUT

/content/arun_similarity/sample_similarity_features.csv
/content/arun_similarity/ground_truth_pairs.csv
/content/arun_similarity/feature_comparison.csv

Feature table shape: (10000, 9)
Ground-truth positive pairs: 756838

READY FOR ML INTEGRATION


In [6]:
# ============================================================
# PROPER SIMILARITY VALIDATION
# Creates both POSITIVE and NEGATIVE pairs
# ============================================================

import os
import random
import pandas as pd
import numpy as np

print("=" * 70)
print("CREATING PROPER MATCH / NON-MATCH VALIDATION SAMPLE")
print("=" * 70)

# ------------------------------------------------------------
# 1. Keep only S1 -> S2 ground-truth matches
# ------------------------------------------------------------

positive_pairs = ground_truth_pairs[
    ground_truth_pairs["matched_entity_id"].str.startswith("S2-", na=False)
].copy()

positive_pairs = positive_pairs[
    positive_pairs["s1_entity_id"].isin(s1[ID_COL]) &
    positive_pairs["matched_entity_id"].isin(s2[ID_COL])
]

print("Available positive S1-S2 pairs:", len(positive_pairs))

# ------------------------------------------------------------
# 2. Sample positive pairs
# ------------------------------------------------------------

N_POSITIVE = min(500, len(positive_pairs))

positive_sample = positive_pairs.sample(
    n=N_POSITIVE,
    random_state=42
).copy()

positive_sample["is_match"] = 1

print("Positive sample:", len(positive_sample))

# ------------------------------------------------------------
# 3. Create negative pairs
# ------------------------------------------------------------

s1_ids = s1[ID_COL].tolist()
s2_ids = s2[ID_COL].tolist()

# Ground-truth lookup
true_matches = {}

for _, row in positive_pairs.iterrows():
    s1_id = row["s1_entity_id"]
    s2_id = row["matched_entity_id"]

    if s1_id not in true_matches:
        true_matches[s1_id] = set()

    true_matches[s1_id].add(s2_id)

negative_rows = []

rng = np.random.default_rng(42)

for _, row in positive_sample.iterrows():

    s1_id = row["s1_entity_id"]

    # Keep selecting until we find a known non-match
    while True:

        random_s2 = s2_ids[
            rng.integers(0, len(s2_ids))
        ]

        if random_s2 not in true_matches.get(s1_id, set()):
            break

    negative_rows.append({
        "s1_entity_id": s1_id,
        "matched_entity_id": random_s2,
        "is_match": 0
    })

negative_sample = pd.DataFrame(negative_rows)

print("Negative sample:", len(negative_sample))

# ------------------------------------------------------------
# 4. Combine positive + negative pairs
# ------------------------------------------------------------

validation_pairs = pd.concat(
    [
        positive_sample[
            ["s1_entity_id", "matched_entity_id", "is_match"]
        ],
        negative_sample[
            ["s1_entity_id", "matched_entity_id", "is_match"]
        ]
    ],
    ignore_index=True
)

# Shuffle
validation_pairs = validation_pairs.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("\nValidation pairs:", len(validation_pairs))
print("\nClass distribution:")
print(validation_pairs["is_match"].value_counts())

# ------------------------------------------------------------
# 5. Create lookup dictionaries
# ------------------------------------------------------------

s1_lookup = s1.set_index(ID_COL).to_dict("index")
s2_lookup = s2.set_index(ID_COL).to_dict("index")

# ------------------------------------------------------------
# 6. Calculate similarity features
# ------------------------------------------------------------

feature_rows = []

for _, pair in validation_pairs.iterrows():

    s1_id = pair["s1_entity_id"]
    s2_id = pair["matched_entity_id"]

    row1 = s1_lookup[s1_id]
    row2 = s2_lookup[s2_id]

    features = calculate_similarity_features(
        row1[NAME_COL],
        row2[NAME_COL],

        row1[ADDRESS_COL],
        row2[ADDRESS_COL],

        row1[COUNTRY_COL],
        row2[COUNTRY_COL]
    )

    feature_rows.append({
        "s1_entity_id": s1_id,
        "s2_entity_id": s2_id,

        **features,

        "is_match": pair["is_match"]
    })

validation_features = pd.DataFrame(feature_rows)

# ------------------------------------------------------------
# 7. Check result
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("VALIDATION FEATURE TABLE")
print("=" * 70)

print("Shape:", validation_features.shape)

print("\nFirst 10 rows:")
display(validation_features.head(10))

# ------------------------------------------------------------
# 8. Missing value check
# ------------------------------------------------------------

print("\nMissing values:")
print(validation_features.isnull().sum())

# ------------------------------------------------------------
# 9. Match vs non-match comparison
# ------------------------------------------------------------

feature_columns = [
    "name_jaccard",
    "name_levenshtein",
    "name_tfidf_cosine",
    "address_jaccard",
    "address_levenshtein",
    "address_tfidf_cosine",
    "country_match"
]

comparison = validation_features.groupby(
    "is_match"
)[feature_columns].mean()

print("\n" + "=" * 70)
print("MATCH VS NON-MATCH FEATURE COMPARISON")
print("=" * 70)

display(comparison)

# ------------------------------------------------------------
# 10. Save validation dataset
# ------------------------------------------------------------

OUTPUT_DIR = "/content/arun_similarity"
os.makedirs(OUTPUT_DIR, exist_ok=True)

validation_features.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "validated_similarity_features.csv"
    ),
    index=False
)

comparison.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "validated_feature_comparison.csv"
    )
)

print("\n" + "=" * 70)
print("VALIDATION COMPLETED")
print("=" * 70)

print(f"""
Positive pairs : {(validation_features["is_match"] == 1).sum()}
Negative pairs : {(validation_features["is_match"] == 0).sum()}
Total pairs    : {len(validation_features)}

Saved:
✓ {OUTPUT_DIR}/validated_similarity_features.csv
✓ {OUTPUT_DIR}/validated_feature_comparison.csv
""")

CREATING PROPER MATCH / NON-MATCH VALIDATION SAMPLE
Available positive S1-S2 pairs: 519
Positive sample: 500
Negative sample: 500

Validation pairs: 1000

Class distribution:
is_match
0    500
1    500
Name: count, dtype: int64

VALIDATION FEATURE TABLE
Shape: (1000, 10)

First 10 rows:


,s1_entity_id,s2_entity_id,name_jaccard,name_levenshtein,name_tfidf_cosine,address_jaccard,address_levenshtein,address_tfidf_cosine,country_match,is_match
0,S1-867508043,S2-791708103,0.0,0.307692,0.000000,0.000000,0.400000,0.000000,1,0
1,S1-114983362,S2-956300503,0.0,0.428571,0.000000,0.000000,0.413793,0.000000,1,0
2,S1-794206777,S2-885293462,0.0,0.071429,0.000000,0.000000,0.285714,0.000000,0,0
3,S1-592863769,S2-782027568,0.0,0.304348,0.000000,0.000000,0.261682,0.000000,0,0
4,S1-719532580,S2-392745675,0.0,0.869565,0.000000,0.833333,0.539683,0.678755,1,1
5,S1-582097617,S2-324123305,0.0,0.347826,0.000000,0.000000,0.357724,0.000000,1,0
6,S1-850422338,S2-594722602,0.0,0.333333,0.000000,0.000000,0.270833,0.000000,0,0
7,S1-600765503,S2-576789780,0.0,0.250000,0.000000,0.000000,0.291667,0.000000,1,0
8,S1-872690724,S2-21245594,0.0,0.263158,0.000000,0.000000,0.315789,0.000000,0,0
9,S1-706591875,S2-811072158,1.0,0.861538,0.849577,0.400000,0.753623,0.308805,1,1



Missing values:
s1_entity_id            0
s2_entity_id            0
name_jaccard            0
name_levenshtein        0
name_tfidf_cosine       0
address_jaccard         0
address_levenshtein     0
address_tfidf_cosine    0
country_match           0
is_match                0
dtype: int64

MATCH VS NON-MATCH FEATURE COMPARISON


,name_jaccard,name_levenshtein,name_tfidf_cosine,address_jaccard,address_levenshtein,address_tfidf_cosine,country_match
is_match,,,,,,,
0,0.009235,0.304564,0.005421,0.008207,0.311809,0.004783,0.476
1,0.614715,0.792264,0.529721,0.687921,0.791009,0.589969,1.000



VALIDATION COMPLETED

Positive pairs : 500
Negative pairs : 500
Total pairs    : 1000

Saved:
✓ /content/arun_similarity/validated_similarity_features.csv
✓ /content/arun_similarity/validated_feature_comparison.csv



In [7]:
import pandas as pd

features = pd.read_csv(
    "/content/arun_similarity/validated_similarity_features.csv"
)

print("Shape:", features.shape)
print("\nColumns:")
print(features.columns.tolist())

print("\nClass distribution:")
print(features["is_match"].value_counts())

display(features.head())

Shape: (1000, 10)

Columns:
['s1_entity_id', 's2_entity_id', 'name_jaccard', 'name_levenshtein', 'name_tfidf_cosine', 'address_jaccard', 'address_levenshtein', 'address_tfidf_cosine', 'country_match', 'is_match']

Class distribution:
is_match
0    500
1    500
Name: count, dtype: int64


,s1_entity_id,s2_entity_id,name_jaccard,name_levenshtein,name_tfidf_cosine,address_jaccard,address_levenshtein,address_tfidf_cosine,country_match,is_match
0,S1-867508043,S2-791708103,0.0,0.307692,0.0,0.000000,0.400000,0.000000,1,0
1,S1-114983362,S2-956300503,0.0,0.428571,0.0,0.000000,0.413793,0.000000,1,0
2,S1-794206777,S2-885293462,0.0,0.071429,0.0,0.000000,0.285714,0.000000,0,0
3,S1-592863769,S2-782027568,0.0,0.304348,0.0,0.000000,0.261682,0.000000,0,0
4,S1-719532580,S2-392745675,0.0,0.869565,0.0,0.833333,0.539683,0.678755,1,1


In [9]:
import pandas as pd

features = pd.read_csv(
    "/content/arun_similarity/validated_similarity_features.csv"
)

# Keep only ML features + label
feature_cols = [
    "name_jaccard",
    "name_levenshtein",
    "name_tfidf_cosine",
    "address_jaccard",
    "address_levenshtein",
    "address_tfidf_cosine",
    "country_match",
    "is_match"
]

final_features = features[feature_cols].copy()

print(final_features.head())
print("\nShape:", final_features.shape)

   name_jaccard  name_levenshtein  name_tfidf_cosine  address_jaccard  \
0           0.0          0.307692                0.0         0.000000   
1           0.0          0.428571                0.0         0.000000   
2           0.0          0.071429                0.0         0.000000   
3           0.0          0.304348                0.0         0.000000   
4           0.0          0.869565                0.0         0.833333   

   address_levenshtein  address_tfidf_cosine  country_match  is_match  
0             0.400000              0.000000              1         0  
1             0.413793              0.000000              1         0  
2             0.285714              0.000000              0         0  
3             0.261682              0.000000              0         0  
4             0.539683              0.678755              1         1  

Shape: (1000, 8)


In [10]:
comparison = final_features.groupby("is_match").mean()

display(comparison)

,name_jaccard,name_levenshtein,name_tfidf_cosine,address_jaccard,address_levenshtein,address_tfidf_cosine,country_match
is_match,,,,,,,
0,0.009235,0.304564,0.005421,0.008207,0.311809,0.004783,0.476
1,0.614715,0.792264,0.529721,0.687921,0.791009,0.589969,1.000


In [12]:
import os

matches = []

for root, dirs, files in os.walk("/content"):
    if "candidate_pairs.csv" in files:
        matches.append(os.path.join(root, "candidate_pairs.csv"))

print("Found files:")
for path in matches:
    size_gb = os.path.getsize(path) / (1024**3)
    print(path, "→", round(size_gb, 2), "GB")

Found files:


In [14]:
import os

candidate_path = "/content/candidate_pairs.csv"

print("Exists:", os.path.exists(candidate_path))

if os.path.exists(candidate_path):
    size_gb = os.path.getsize(candidate_path) / (1024**3)
    print("File size:", round(size_gb, 2), "GB")

Exists: True
File size: 0.01 GB


In [15]:
import pandas as pd

sample = pd.read_csv(
    candidate_path,
    nrows=10
)

print("Columns:")
print(sample.columns.tolist())

print("\nShape:")
print(sample.shape)

display(sample)

Columns:
['s1_id', 'matched_id', 'matched_source', 'blocking_rule']

Shape:
(10, 4)


,s1_id,matched_id,matched_source,blocking_rule
0,S1-506318355,S3-713712067,S3,B1
1,S1-123026218,S2-295504750,S2,B1
2,S1-191014456,S2-632259122,S2,B1|B2
3,S1-8676749,S2-823569314,S2,B1
4,S1-317175025,S3-377721037,S3,B1
5,S1-57549224,S2-810964374,S2,B1
6,S1-238631953,S3-450969569,S3,B1
7,S1-833576891,S3-41278252,S3,B1
8,S1-848194269,S2-373347562,S2,B1
9,S1-468215380,S2-36301933,S2,B1


In [16]:
import os

source_files = [
    "/content/train_source1.tsv",
    "/content/train_source2.tsv",
    "/content/train_source3.tsv"
]

for path in source_files:
    print("\n", path)
    print("Exists:", os.path.exists(path))

    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024**2)
        print("Size:", round(size_mb, 2), "MB")


 /content/train_source1.tsv
Exists: True
Size: 186.0 MB

 /content/train_source2.tsv
Exists: True
Size: 199.0 MB

 /content/train_source3.tsv
Exists: True
Size: 205.0 MB


In [17]:
import pandas as pd

for path in source_files:
    df = pd.read_csv(path, sep="\t", nrows=3)
    print("\n", path)
    print(df.columns.tolist())
    display(df)


 /content/train_source1.tsv
['entity_id', 'business_name', 'business_address', 'country']


,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US



 /content/train_source2.tsv
['entity_id', 'business_name', 'business_address', 'country']


,entity_id,business_name,business_address,country
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India



 /content/train_source3.tsv
['entity_id', 'business_name', 'business_address', 'country']


,entity_id,business_name,business_address,country
0,S3-202863386,wilfordhancock.com,"Mack Rd, Haltom City, Texas",US
1,S3-859268022,International South Consultants Private Ltd,NaN,India
2,S3-22467283,LLC Moncada Léarning Center,"5780 Fawn Ct, Fort Worth, Texas",US


In [18]:
import pandas as pd

s1_path = "/content/train_source1.tsv"
s2_path = "/content/train_source2.tsv"
s3_path = "/content/train_source3.tsv"

s1 = pd.read_csv(s1_path, sep="\t")
s2 = pd.read_csv(s2_path, sep="\t")
s3 = pd.read_csv(s3_path, sep="\t")

print("S1:", s1.shape)
print("S2:", s2.shape)
print("S3:", s3.shape)

UnicodeDecodeError: 'utf-8' codec can't decode bytes in position 0-1: unexpected end of data

In [19]:
s3_path = "/content/train_source3.tsv"

with open(s3_path, "rb") as f:
    raw = f.read(100)

print(raw)

b'entity_id\tbusiness_name\tbusiness_address\tcountry\nS3-202863386\twilfordhancock.com\tMack Rd, Haltom Cit'


In [20]:
import chardet

with open(s3_path, "rb") as f:
    raw = f.read(100000)

result = chardet.detect(raw)

print(result)

{'encoding': 'utf-8', 'confidence': 0.99, 'language': ''}


In [21]:
s3 = pd.read_csv(
    s3_path,
    sep="\t",
    encoding="utf-8-sig"
)

print("S3:", s3.shape)
print(s3.columns.tolist())

S3: (5285603, 4)
['entity_id', 'business_name', 'business_address', 'country']


In [23]:
print("S1:", s1.shape)
print("S2:", s2.shape)
print("S3:", s3.shape)

S1: (2103974, 4)
S2: (2201690, 4)
S3: (5285603, 4)


In [24]:
candidate_path = "/content/candidate_pairs.csv"

candidates_test = pd.read_csv(
    candidate_path,
    nrows=10000
)

print("Candidate rows:", len(candidates_test))
display(candidates_test.head())

Candidate rows: 10000


,s1_id,matched_id,matched_source,blocking_rule
0,S1-506318355,S3-713712067,S3,B1
1,S1-123026218,S2-295504750,S2,B1
2,S1-191014456,S2-632259122,S2,B1|B2
3,S1-8676749,S2-823569314,S2,B1
4,S1-317175025,S3-377721037,S3,B1


In [25]:
s1_ids = set(s1["entity_id"])
s2_ids = set(s2["entity_id"])
s3_ids = set(s3["entity_id"])

print("S1 candidate IDs found:",
      candidates_test["s1_id"].isin(s1_ids).sum(),
      "/", len(candidates_test))

s2_mask = (
    (candidates_test["matched_source"] == "S2") &
    candidates_test["matched_id"].isin(s2_ids)
)

s3_mask = (
    (candidates_test["matched_source"] == "S3") &
    candidates_test["matched_id"].isin(s3_ids)
)

print("S2 candidate IDs found:", s2_mask.sum())
print("S3 candidate IDs found:", s3_mask.sum())

S1 candidate IDs found: 10000 / 10000
S2 candidate IDs found: 4857
S3 candidate IDs found: 5143


In [26]:
import pandas as pd
import numpy as np
import re

from rapidfuzz.fuzz import ratio, token_set_ratio


# -----------------------------
# Text cleaning
# -----------------------------
def clean_text(value):
    if pd.isna(value):
        return ""

    value = str(value).lower()
    value = re.sub(r"\s+", " ", value)
    return value.strip()


# -----------------------------
# Similarity functions
# -----------------------------
def jaccard_similarity(a, b):
    a = set(clean_text(a).split())
    b = set(clean_text(b).split())

    if not a or not b:
        return 0.0

    return len(a & b) / len(a | b)


def levenshtein_similarity(a, b):
    a = clean_text(a)
    b = clean_text(b)

    if not a or not b:
        return 0.0

    return ratio(a, b) / 100


def token_similarity(a, b):
    a = clean_text(a)
    b = clean_text(b)

    if not a or not b:
        return 0.0

    return token_set_ratio(a, b) / 100


def country_match(a, b):
    a = clean_text(a)
    b = clean_text(b)

    if not a or not b:
        return 0

    return int(a == b)

In [27]:
# Make lookups
s1_lookup = s1.set_index("entity_id")
s2_lookup = s2.set_index("entity_id")
s3_lookup = s3.set_index("entity_id")


feature_rows = []

for _, row in candidates_test.iterrows():

    s1_id = row["s1_id"]
    matched_id = row["matched_id"]
    source = row["matched_source"]

    # Get S1 record
    r1 = s1_lookup.loc[s1_id]

    # Get S2 or S3 record
    if source == "S2":
        r2 = s2_lookup.loc[matched_id]
    else:
        r2 = s3_lookup.loc[matched_id]

    feature_rows.append({
        "s1_id": s1_id,
        "matched_id": matched_id,
        "matched_source": source,
        "blocking_rule": row["blocking_rule"],

        "name_jaccard":
            jaccard_similarity(
                r1["business_name"],
                r2["business_name"]
            ),

        "name_levenshtein":
            levenshtein_similarity(
                r1["business_name"],
                r2["business_name"]
            ),

        "name_token_similarity":
            token_similarity(
                r1["business_name"],
                r2["business_name"]
            ),

        "address_jaccard":
            jaccard_similarity(
                r1["business_address"],
                r2["business_address"]
            ),

        "address_levenshtein":
            levenshtein_similarity(
                r1["business_address"],
                r2["business_address"]
            ),

        "address_token_similarity":
            token_similarity(
                r1["business_address"],
                r2["business_address"]
            ),

        "country_match":
            country_match(
                r1["country"],
                r2["country"]
            )
    })


features_10k = pd.DataFrame(feature_rows)

print("Feature dataset shape:", features_10k.shape)

display(features_10k.head())

Feature dataset shape: (10000, 11)


,s1_id,matched_id,matched_source,blocking_rule,name_jaccard,name_levenshtein,name_token_similarity,address_jaccard,address_levenshtein,address_token_similarity,country_match
0,S1-506318355,S3-713712067,S3,B1,0.166667,0.55814,0.666667,0.000000,0.321429,0.321429,1
1,S1-123026218,S2-295504750,S2,B1,0.400000,0.72000,0.800000,0.045455,0.375758,0.410256,1
2,S1-191014456,S2-632259122,S2,B1|B2,0.200000,0.45000,0.500000,0.043478,0.408163,0.489796,1
3,S1-8676749,S2-823569314,S2,B1,0.000000,0.40000,0.352941,0.000000,0.376812,0.405797,1
4,S1-317175025,S3-377721037,S3,B1,0.125000,0.40000,0.509091,0.000000,0.380952,0.380952,1


In [28]:
output_path = "/content/feature_engineered_10k.csv"

features_10k.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)
print("Rows:", len(features_10k))

Saved: /content/feature_engineered_10k.csv
Rows: 10000


In [29]:
gt_path = "/content/train_ground_truth.tsv"

gt = pd.read_csv(
    gt_path,
    sep="\t"
)

print("Ground truth shape:", gt.shape)
print(gt.columns.tolist())

display(gt.head())

Ground truth shape: (2206821, 2)
['source1_entity_id', 'matched_entity_ids']


,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."


In [30]:
ground_truth_map = {}

for _, row in gt.iterrows():

    s1_id = row["source1_entity_id"]

    matched_ids = str(row["matched_entity_ids"]).split(",")

    ground_truth_map[s1_id] = set(
        x.strip() for x in matched_ids if x.strip()
    )

print("S1 entities in ground truth:", len(ground_truth_map))

S1 entities in ground truth: 2206821


In [31]:
def get_label(row):

    s1_id = row["s1_id"]
    matched_id = row["matched_id"]

    true_matches = ground_truth_map.get(s1_id, set())

    return int(matched_id in true_matches)


features_10k["is_match"] = features_10k.apply(
    get_label,
    axis=1
)

print(features_10k["is_match"].value_counts())

is_match
0    9960
1      40
Name: count, dtype: int64


In [32]:
feature_cols = [
    "name_jaccard",
    "name_levenshtein",
    "name_token_similarity",
    "address_jaccard",
    "address_levenshtein",
    "address_token_similarity",
    "country_match"
]

comparison = features_10k.groupby("is_match")[feature_cols].mean()

display(comparison)

,name_jaccard,name_levenshtein,name_token_similarity,address_jaccard,address_levenshtein,address_token_similarity,country_match
is_match,,,,,,,
0,0.072418,0.391176,0.408635,0.061541,0.413620,0.437649,0.972088
1,0.590298,0.802671,0.896235,0.575115,0.776187,0.891466,1.000000


In [33]:
output_path = "/content/feature_engineered_10k_labelled.csv"

features_10k.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)
print("Shape:", features_10k.shape)

Saved: /content/feature_engineered_10k_labelled.csv
Shape: (10000, 12)


In [34]:
print("Positive pairs:", (features_10k["is_match"] == 1).sum())
print("Negative pairs:", (features_10k["is_match"] == 0).sum())

print("\nPositive ratio:")
print(features_10k["is_match"].mean())

Positive pairs: 40
Negative pairs: 9960

Positive ratio:
0.004


In [35]:
positive_pairs = features_10k[
    features_10k["is_match"] == 1
].copy()

print("Positive pairs:", len(positive_pairs))

display(
    positive_pairs[
        [
            "s1_id",
            "matched_id",
            "matched_source",
            "blocking_rule",
            "name_jaccard",
            "name_levenshtein",
            "name_token_similarity",
            "address_jaccard",
            "address_levenshtein",
            "address_token_similarity",
            "country_match"
        ]
    ].head(20)
)

Positive pairs: 40


,s1_id,matched_id,matched_source,blocking_rule,name_jaccard,name_levenshtein,name_token_similarity,address_jaccard,address_levenshtein,address_token_similarity,country_match
22,S1-934688692,S3-988309567,S3,B1|B3,0.600000,0.813559,0.941176,0.812500,0.947368,0.984615,1
77,S1-225158067,S3-675606143,S3,B1|B3,0.750000,0.900000,1.000000,0.666667,0.606061,0.947368,1
868,S1-140674772,S2-171243022,S2,B1|B2|B3,0.500000,0.545455,0.750000,0.571429,0.916667,0.888889,1
1285,S1-765765749,S3-321988278,S3,B1|B2,0.666667,0.809524,1.000000,0.000000,0.000000,0.000000,1
1350,S1-269884741,S3-737904052,S3,B1|B3,0.400000,0.428571,1.000000,0.500000,0.525000,0.975000,1
2014,S1-853644648,S2-585083439,S2,B1|B2|B3,1.000000,0.625000,1.000000,0.500000,0.550725,0.927536,1
2019,S1-533227023,S3-169638161,S3,B1|B2|B3,0.500000,0.666667,0.833333,0.777778,0.900901,0.969072,1
3063,S1-131976951,S3-7812799,S3,B1|B2|B3,1.000000,1.000000,1.000000,0.250000,0.805970,0.805970,1
3885,S1-464085750,S2-714179024,S2,B1|B2|B3,1.000000,1.000000,1.000000,0.454545,0.511278,0.909091,1
4334,S1-933884440,S2-701210536,S2,B1|B3,0.714286,0.976744,0.976744,0.500000,0.791045,0.903846,1


In [36]:
from rapidfuzz.fuzz import token_set_ratio

def normalized_length_diff(a, b):
    a = clean_text(a)
    b = clean_text(b)

    if not a or not b:
        return 1.0

    max_len = max(len(a), len(b))

    return abs(len(a) - len(b)) / max_len


# Create lookup records
def get_record_fast(entity_id, source):
    if source == "S2":
        return s2_lookup.loc[entity_id]
    else:
        return s3_lookup.loc[entity_id]


name_token_set_values = []
address_token_set_values = []
name_length_diff_values = []
address_length_diff_values = []

for _, row in candidates_test.iterrows():

    r1 = s1_lookup.loc[row["s1_id"]]
    r2 = get_record_fast(
        row["matched_id"],
        row["matched_source"]
    )

    name_token_set_values.append(
        token_set_ratio(
            clean_text(r1["business_name"]),
            clean_text(r2["business_name"])
        ) / 100
    )

    address_token_set_values.append(
        token_set_ratio(
            clean_text(r1["business_address"]),
            clean_text(r2["business_address"])
        ) / 100
    )

    name_length_diff_values.append(
        normalized_length_diff(
            r1["business_name"],
            r2["business_name"]
        )
    )

    address_length_diff_values.append(
        normalized_length_diff(
            r1["business_address"],
            r2["business_address"]
        )
    )


features_10k["name_token_set_ratio"] = name_token_set_values
features_10k["address_token_set_ratio"] = address_token_set_values
features_10k["name_length_diff"] = name_length_diff_values
features_10k["address_length_diff"] = address_length_diff_values

print(features_10k.shape)
display(features_10k.head())

(10000, 16)


,s1_id,matched_id,matched_source,blocking_rule,name_jaccard,name_levenshtein,name_token_similarity,address_jaccard,address_levenshtein,address_token_similarity,country_match,is_match,name_token_set_ratio,address_token_set_ratio,name_length_diff,address_length_diff
0,S1-506318355,S3-713712067,S3,B1,0.166667,0.55814,0.666667,0.000000,0.321429,0.321429,1,0,0.666667,0.321429,0.280000,0.193548
1,S1-123026218,S2-295504750,S2,B1,0.400000,0.72000,0.800000,0.045455,0.375758,0.410256,1,0,0.800000,0.410256,0.275862,0.146067
2,S1-191014456,S2-632259122,S2,B1|B2,0.200000,0.45000,0.500000,0.043478,0.408163,0.489796,1,0,0.500000,0.489796,0.333333,0.380165
3,S1-8676749,S2-823569314,S2,B1,0.000000,0.40000,0.352941,0.000000,0.376812,0.405797,1,0,0.352941,0.405797,0.333333,0.230769
4,S1-317175025,S3-377721037,S3,B1,0.125000,0.40000,0.509091,0.000000,0.380952,0.380952,1,0,0.509091,0.380952,0.103448,0.090909


In [37]:
new_features = [
    "name_token_set_ratio",
    "address_token_set_ratio",
    "name_length_diff",
    "address_length_diff"
]

display(
    features_10k.groupby("is_match")[new_features].mean()
)

,name_token_set_ratio,address_token_set_ratio,name_length_diff,address_length_diff
is_match,,,,
0,0.408635,0.437649,0.297941,0.234549
1,0.896235,0.891466,0.145331,0.162639


In [38]:
output_path = "/content/feature_engineered_10k_V2_labelled.csv"

features_10k.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)
print("Shape:", features_10k.shape)
print("Columns:")
print(features_10k.columns.tolist())

Saved: /content/feature_engineered_10k_V2_labelled.csv
Shape: (10000, 16)
Columns:
['s1_id', 'matched_id', 'matched_source', 'blocking_rule', 'name_jaccard', 'name_levenshtein', 'name_token_similarity', 'address_jaccard', 'address_levenshtein', 'address_token_similarity', 'country_match', 'is_match', 'name_token_set_ratio', 'address_token_set_ratio', 'name_length_diff', 'address_length_diff']


In [39]:
!pip install -q pyarrow

In [40]:
!pip install -q duckdb pyarrow rapidfuzz

In [41]:
import os

files_to_check = [
    "/content/candidate_pairs.csv",
    "/content/train_source1.tsv",
    "/content/train_source2.tsv",
    "/content/train_source3.tsv",
    "/content/train_ground_truth.tsv"
]

for f in files_to_check:
    print(f, "->", os.path.exists(f))

/content/candidate_pairs.csv -> True
/content/train_source1.tsv -> True
/content/train_source2.tsv -> True
/content/train_source3.tsv -> True
/content/train_ground_truth.tsv -> True


In [42]:
import pandas as pd
import numpy as np
from rapidfuzz.fuzz import ratio, token_set_ratio
import os

# =========================
# PATHS
# =========================
candidate_path = "/content/candidate_pairs.csv"
s1_path = "/content/train_source1.tsv"
s2_path = "/content/train_source2.tsv"
s3_path = "/content/train_source3.tsv"

# =========================
# LOAD SOURCE DATA
# =========================
print("Loading source files...")

s1 = pd.read_csv(s1_path, sep="\t")
s2 = pd.read_csv(s2_path, sep="\t")
s3 = pd.read_csv(s3_path, sep="\t", encoding="utf-8-sig")

print("S1:", s1.shape)
print("S2:", s2.shape)
print("S3:", s3.shape)

# =========================
# CREATE LOOKUPS
# =========================
s1_lookup = s1.set_index("entity_id").to_dict("index")
s2_lookup = s2.set_index("entity_id").to_dict("index")
s3_lookup = s3.set_index("entity_id").to_dict("index")

print("Lookups created.")

# =========================
# TEXT CLEANING
# =========================
def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).lower().strip()

# =========================
# FEATURE FUNCTION
# =========================
def make_features(s1_row, matched_row):

    name1 = clean_text(s1_row["business_name"])
    name2 = clean_text(matched_row["business_name"])

    addr1 = clean_text(s1_row["business_address"])
    addr2 = clean_text(matched_row["business_address"])

    country1 = clean_text(s1_row["country"])
    country2 = clean_text(matched_row["country"])

    return {
        "name_levenshtein":
            ratio(name1, name2) / 100,

        "name_token_similarity":
            token_set_ratio(name1, name2) / 100,

        "address_levenshtein":
            ratio(addr1, addr2) / 100,

        "address_token_similarity":
            token_set_ratio(addr1, addr2) / 100,

        "name_length_diff":
            abs(len(name1) - len(name2)),

        "address_length_diff":
            abs(len(addr1) - len(addr2)),

        "country_match":
            int(country1 == country2)
    }

# =========================
# READ ONLY 100K CANDIDATES
# =========================
print("\nReading first 100,000 candidate pairs...")

candidates = pd.read_csv(
    candidate_path,
    nrows=100_000
)

print("Candidate rows:", len(candidates))
print(candidates.head())

# =========================
# FEATURE ENGINEERING
# =========================
feature_rows = []

for i, row in candidates.iterrows():

    s1_id = row["s1_id"]
    matched_id = row["matched_id"]
    source = row["matched_source"]

    s1_row = s1_lookup.get(s1_id)

    if source == "S2":
        matched_row = s2_lookup.get(matched_id)
    else:
        matched_row = s3_lookup.get(matched_id)

    if s1_row is None or matched_row is None:
        continue

    features = make_features(s1_row, matched_row)

    features["s1_id"] = s1_id
    features["matched_id"] = matched_id
    features["matched_source"] = source
    features["blocking_rule"] = row["blocking_rule"]

    feature_rows.append(features)

features_100k = pd.DataFrame(feature_rows)

print("\n==============================")
print("FEATURE ENGINEERING COMPLETE")
print("==============================")

print("Shape:", features_100k.shape)

print("\nColumns:")
print(features_100k.columns.tolist())

display(features_100k.head())

Loading source files...
S1: (2206821, 4)
S2: (5034616, 4)
S3: (5285603, 4)
Lookups created.

Reading first 100,000 candidate pairs...
Candidate rows: 100000
          s1_id    matched_id matched_source blocking_rule
0  S1-506318355  S3-713712067             S3            B1
1  S1-123026218  S2-295504750             S2            B1
2  S1-191014456  S2-632259122             S2         B1|B2
3    S1-8676749  S2-823569314             S2            B1
4  S1-317175025  S3-377721037             S3            B1

FEATURE ENGINEERING COMPLETE
Shape: (100000, 11)

Columns:
['name_levenshtein', 'name_token_similarity', 'address_levenshtein', 'address_token_similarity', 'name_length_diff', 'address_length_diff', 'country_match', 's1_id', 'matched_id', 'matched_source', 'blocking_rule']


,name_levenshtein,name_token_similarity,address_levenshtein,address_token_similarity,name_length_diff,address_length_diff,country_match,s1_id,matched_id,matched_source,blocking_rule
0,0.55814,0.666667,0.321429,0.321429,7,6,1,S1-506318355,S3-713712067,S3,B1
1,0.72000,0.800000,0.375758,0.410256,8,13,1,S1-123026218,S2-295504750,S2,B1
2,0.45000,0.500000,0.408163,0.489796,8,46,1,S1-191014456,S2-632259122,S2,B1|B2
3,0.40000,0.352941,0.371429,0.405797,8,10,1,S1-8676749,S2-823569314,S2,B1
4,0.40000,0.509091,0.380952,0.380952,3,4,1,S1-317175025,S3-377721037,S3,B1


In [43]:
# ==========================================
# LOAD GROUND TRUTH
# ==========================================

gt_path = "/content/train_ground_truth.tsv"

print("Loading ground truth...")

gt = pd.read_csv(
    gt_path,
    sep="\t"
)

print("Ground truth shape:", gt.shape)

# Create dictionary:
# S1 ID -> set of true matched S2/S3 IDs

ground_truth_map = {}

for _, row in gt.iterrows():

    s1_id = str(row["source1_entity_id"]).strip()

    matched_ids = str(row["matched_entity_ids"]).split(",")

    ground_truth_map[s1_id] = {
        x.strip()
        for x in matched_ids
        if x.strip()
    }

print("Ground truth S1 entities:", len(ground_truth_map))


# ==========================================
# ADD LABEL
# ==========================================

def get_label(row):

    s1_id = row["s1_id"]
    matched_id = row["matched_id"]

    true_matches = ground_truth_map.get(s1_id, set())

    return int(matched_id in true_matches)


features_100k["is_match"] = features_100k.apply(
    get_label,
    axis=1
)


# ==========================================
# CHECK
# ==========================================

print("\n==============================")
print("LABELING COMPLETE")
print("==============================")

print(
    features_100k["is_match"].value_counts()
)

print("\nPositive ratio:")
print(features_100k["is_match"].mean())

display(
    features_100k.head()
)

Loading ground truth...
Ground truth shape: (2206821, 2)
Ground truth S1 entities: 2206821

LABELING COMPLETE
is_match
0    99746
1      254
Name: count, dtype: int64

Positive ratio:
0.00254


,name_levenshtein,name_token_similarity,address_levenshtein,address_token_similarity,name_length_diff,address_length_diff,country_match,s1_id,matched_id,matched_source,blocking_rule,is_match
0,0.55814,0.666667,0.321429,0.321429,7,6,1,S1-506318355,S3-713712067,S3,B1,0
1,0.72000,0.800000,0.375758,0.410256,8,13,1,S1-123026218,S2-295504750,S2,B1,0
2,0.45000,0.500000,0.408163,0.489796,8,46,1,S1-191014456,S2-632259122,S2,B1|B2,0
3,0.40000,0.352941,0.371429,0.405797,8,10,1,S1-8676749,S2-823569314,S2,B1,0
4,0.40000,0.509091,0.380952,0.380952,3,4,1,S1-317175025,S3-377721037,S3,B1,0


In [44]:
output_path = "/content/feature_engineered_100k_V2_labelled.csv"

features_100k.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)
print("Shape:", features_100k.shape)
print("Positive:", (features_100k["is_match"] == 1).sum())
print("Negative:", (features_100k["is_match"] == 0).sum())

Saved: /content/feature_engineered_100k_V2_labelled.csv
Shape: (100000, 12)
Positive: 254
Negative: 99746


In [45]:
check = pd.read_csv(output_path)

print(check.shape)
print(check["is_match"].value_counts())
display(check[check["is_match"] == 1].head(10))

(100000, 12)
is_match
0    99746
1      254
Name: count, dtype: int64


,name_levenshtein,name_token_similarity,address_levenshtein,address_token_similarity,name_length_diff,address_length_diff,country_match,s1_id,matched_id,matched_source,blocking_rule,is_match
22,0.813559,0.941176,0.947368,0.984615,5,11,1,S1-934688692,S3-988309567,S3,B1|B3,1
77,0.878049,1.000000,0.606061,0.947368,5,6,1,S1-225158067,S3-675606143,S3,B1|B3,1
868,0.545455,0.750000,0.916667,0.888889,3,6,1,S1-140674772,S2-171243022,S2,B1|B2|B3,1
1285,0.809524,1.000000,0.000000,0.000000,8,74,1,S1-765765749,S3-321988278,S3,B1|B2,1
1350,0.428571,1.000000,0.525000,0.975000,24,0,1,S1-269884741,S3-737904052,S3,B1|B3,1
2014,0.625000,1.000000,0.550725,0.927536,0,3,1,S1-853644648,S2-585083439,S2,B1|B2|B3,1
2019,0.666667,0.833333,0.900901,0.969072,2,3,1,S1-533227023,S3-169638161,S3,B1|B2|B3,1
3063,0.984127,1.000000,0.805970,0.805970,1,5,1,S1-131976951,S3-7812799,S3,B1|B2|B3,1
3885,1.000000,1.000000,0.511278,0.909091,0,3,1,S1-464085750,S2-714179024,S2,B1|B2|B3,1
4334,0.976744,0.976744,0.791045,0.903846,0,20,1,S1-933884440,S2-701210536,S2,B1|B3,1


In [46]:
from google.colab import files

files.download("/content/feature_engineered_100k_V2_labelled.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>